# Primitive Description & Paraphrase Authoring

Two routes, auto-detected per primitive from whether `DISPLAY_NAME` and `DESCRIPTION` are already present:

- **Route A — full authoring** (display name / description ABSENT): the LLM produces a `display_name`, a `master` description, and K stylized paraphrases (headline or pure-paraphrase style). The CSV `DISPLAY_NAME`/`DESCRIPTION` are filled, and the master+paraphrases are written to the side file.
- **Route B — paraphrases only** (display name AND description PRESENT): the LLM is given the existing master and only generates K stylized paraphrases. The CSV is untouched; only the side file is written.

Side file: JSON-lines keyed by a stable primitive **hash** (`sha1` of the primitive path), so paraphrases join back to the CSV regardless of route. Style is toggled by `HEADLINE_STYLE`.

Structured output:
- Route A: `{ "display_name": str, "master": str, "paraphrases": [str x K] }`
- Route B: `{ "paraphrases": [str x K] }`


In [2]:
import os, csv, json, time, hashlib
from dotenv import load_dotenv; load_dotenv()
from collections import defaultdict
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel, conlist

# ---- config ----
# TAXONOMY_NAME = "Evergreen_v5"
TAXONOMY_NAME = "RavenPack_Edge"
STR_CSV_PATH = f"{TAXONOMY_NAME}_taxonomy.csv"
CSV_PATH  = Path(STR_CSV_PATH)
CSV_OUT   = Path(STR_CSV_PATH.replace(".csv", ".authored.csv"))

HEADLINE_STYLE: bool = True # True = headline register, False = semantic paraphrase
K            = 5
MODEL        = "gpt-oss20b"
MAX_RETRIES  = 4
RESUME       = True

_style = "headline" if HEADLINE_STYLE else "semantic"
PARA_OUT = Path(f"{TAXONOMY_NAME}-primitive_{_style}_paraphrases.jsonl")

DISPLAY_COL = "DISPLAY_NAME"
DESC_COL    = "DESCRIPTION"

# ---- FILE-LEVEL ROUTE (decided once, not per row) ----
# If an authored CSV already exists -> ROUTE B (paraphrases only), read authored, no CSV output.
# Else -> require the source CSV -> ROUTE A (author display_name+master+K), write authored CSV.
if CSV_OUT.exists():
    ROUTE = "B"                       # paraphrases only
    READ_PATH = CSV_OUT
elif CSV_PATH.exists():
    ROUTE = "A"                       # full authoring
    READ_PATH = CSV_PATH
# else:
#     raise FileNotFoundError(f"neither authored ({CSV_OUT}) nor source ({CSV_PATH}) CSV found")

client = OpenAI(
    base_url=os.getenv("VERTEX_OLLAMA_URL"),
    api_key=os.getenv("VERTEX_OLLAMA_SECRET"),
)
print("ROUTE    :", ROUTE, "(B=paraphrases-only)" if ROUTE=="B" else "(A=full authoring)")
print("READ_PATH:", READ_PATH)
print("CSV_OUT  :", CSV_OUT, "(written only in Route A)")
print("PARA_OUT :", PARA_OUT)
print("style    :", _style, "| K =", K)


NameError: name 'ROUTE' is not defined

In [3]:
models = client.models.list()
for model in models:
    print(model.id)

APITimeoutError: Request timed out.

### Helpers

In [3]:
# ---- structured output schemas (one per route) ----
class ParaOnly(BaseModel):                      # Route B
    paraphrases: conlist(str, min_length=K, max_length=K)

class FullPrimitive(BaseModel):                 # Route A
    display_name: str
    master: str
    paraphrases: conlist(str, min_length=K, max_length=K)


In [4]:
# ---- system prompts: {headline|paraphrase} x {full|para_only} ----
_HEAD_RULES = (
    "STYLE = NEWS HEADLINE register: terse, concrete, front-loaded, the way a wire/alert/blog writes it. "
    "Vary shape across the K (data-print, quote-led, analyst-led, market-reaction, blunt wire). No definition phrasing.\n"
)
_PARA_RULES = (
    "STYLE = PARAPHRASE register: restate the master with varied vocabulary and surface form, one sentence each, "
    "same meaning, same channel, same pole direction; vary wording not content.\n"
)
_COMMON = (
    "Hierarchy: reservoir > dimension > narrative(with a pole) > sub-mechanism > observability-channel. A PRIMITIVE "
    "(one scoring unit) = narrative x sub-mechanism x observability-channel.\n"
    "RULES:\n"
    "- EVERGREEN: never name a specific event, company, person, product, currency, coin, index, central bank, "
    "country, or era (no COVID, GFC, Brexit, tickers, coin names, dates). Durable mechanism level only; use generic "
    "role words (the central bank, a major exporter, the regulator).\n"
    "- POLE DIRECTION & SEPARABILITY: respect the pole exactly. For a bipolar narrative, mirror the sibling pole ONLY "
    "in channel and subject; make DIRECTIONAL CONTENT maximally distinct (use regime-specific vocabulary, not the "
    "antonym of the other pole) so the two poles land far apart in embedding space.\n"
    "- Stay DISTINCT from the other primitives (sub-mechanisms/channels) of the same narrative (given).\n"
    "- Describe the OBSERVABLE CONTENT of the given channel; do NOT restate the taxonomy slug.\n"
)

def system_prompt(full: bool):
    style = _HEAD_RULES if HEADLINE_STYLE else _PARA_RULES
    if full:
        head = (
            "You author embedding targets for a financial-narrative taxonomy.\n"
            f"TASK: for one primitive, produce a short DISPLAY_NAME (hyphenated slug), a MASTER description "
            f"(one sentence, formal definition register), and exactly K={K} stylized variants.\n"
        )
    else:
        head = (
            "You author additional embedding targets for a financial-narrative taxonomy.\n"
            "You are given the primitive's existing MASTER description.\n"
            f"TASK: produce exactly K={K} stylized variants of it (do NOT change the master).\n"
        )
    return head + _COMMON + style

SYSTEM_FULL = system_prompt(full=True)
SYSTEM_PARA = system_prompt(full=False)
print("full-prompt chars:", len(SYSTEM_FULL), "| para-prompt chars:", len(SYSTEM_PARA))


full-prompt chars: 1413 | para-prompt chars: 1388


In [5]:
def user_prompt(r):
    key=(r["TOPIC"],r["GROUP"],r["CATEGORY"])
    def sib(x):
        if x.get("DISPLAY_NAME"): return x["DISPLAY_NAME"]
        parts=[x.get("ROLE",""),x.get("OBSERVABILITY_CHANNEL","")]
        return "/".join(p for p in parts if p) or x["CATEGORY"]
    sibs=[sib(x) for x in by_narr[key] if pid(x)!=pid(r)]
    mc=mirror_category(r)
    lines=[f"reservoir: {r['TOPIC']}", f"dimension: {r['GROUP']}", f"narrative: {r['CATEGORY']}"]
    if HAS_SUBTYPE:
        lines.append(f"pole: {(r.get('SUB_TYPE') or '') or '(none / intensity)'}")
    if HAS_ROLE:
        lines.append(f"sub-mechanism (ROLE): {r.get('ROLE','')}")
    if HAS_CHANNEL:
        lines.append(f"observability-channel: {r.get('OBSERVABILITY_CHANNEL','')}")
    lines.append(f"other primitives in this narrative: {', '.join(sibs) if sibs else '(none)'}")
    if mc:
        lines.append(f"bipolar mirror pole narrative: {mc} (mirror channel/subject, oppose direction)")
    if ROUTE=="A":
        lines.append(f"Produce display_name, master, and exactly {K} {_style} variants.")
    else:
        lines.append(f"existing master: {r[DESC_COL]}")
        lines.append(f"Produce exactly {K} {_style} variants of the master above.")
    return "\n".join(lines)


In [6]:
def author(r):
    sys = SYSTEM_FULL if ROUTE=="A" else SYSTEM_PARA
    schema = FullPrimitive if ROUTE=="A" else ParaOnly
    last=None
    for _ in range(MAX_RETRIES):
        try:
            resp=client.responses.parse(model=MODEL,
                input=[{"role":"system","content":sys},
                       {"role":"user","content":user_prompt(r)}],
                text_format=schema)
            out=resp.output_parsed
            if out is None or len(out.paraphrases)!=K: raise ValueError("bad parse / K mismatch")
            if ROUTE=="A" and (not out.display_name or not out.master): raise ValueError("missing display_name/master")
            return out
        except Exception as e:
            last=e; time.sleep(1.0)
    raise last


### Load data and process

In [7]:
# ---- load and index (schema: ROLE = sub-mechanism; OBSERVABILITY_CHANNEL optional) ----
with READ_PATH.open(encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    fieldnames = list(reader.fieldnames)
    rows = list(reader)

# RavenPack ROLE-variant strip does NOT apply here: in our schema ROLE == sub-mechanism (kept).
# (RavenPack's own role-variant rows would have been reduced upstream if benchmarking RP.)

for col in (DISPLAY_COL, DESC_COL):
    if col not in fieldnames:
        fieldnames = fieldnames + [col]
        for r in rows: r.setdefault(col, "")

HAS_ROLE    = "ROLE" in fieldnames                      # ROLE = sub-mechanism
HAS_CHANNEL = "OBSERVABILITY_CHANNEL" in fieldnames     # optional
HAS_SUBTYPE = "SUB_TYPE" in fieldnames

# Fixed 7-field path for EVERY taxonomy: a column the vendor file lacks (e.g.
# OBSERVABILITY_CHANNEL for RavenPack) is an empty segment, never dropped, so ids
# are built on one base (same rule as narrative_scoring.primitives.PATH_FIELDS).
PATH_COLS = ["TOPIC","GROUP","TYPE","SUB_TYPE","CATEGORY","ROLE","OBSERVABILITY_CHANNEL"]
def ppath(r): return "/".join((r.get(c) or "") for c in PATH_COLS)
def pid(r):   return hashlib.sha1(ppath(r).encode("utf-8")).hexdigest()

by_narr = defaultdict(list); by_type = defaultdict(list)
for r in rows:
    by_narr[(r["TOPIC"], r["GROUP"], r["CATEGORY"])].append(r)
    by_type[(r["TOPIC"], r["GROUP"], r["TYPE"])].append(r)

def mirror_category(r):
    if not HAS_SUBTYPE or not (r.get("SUB_TYPE") or ""): return None
    sibs = {x["CATEGORY"] for x in by_type[(r["TOPIC"], r["GROUP"], r["TYPE"])] if x["CATEGORY"]!=r["CATEGORY"]}
    return sorted(sibs)[0] if len(sibs)==1 else None

dups = len(rows) - len({ppath(r) for r in rows})
assert dups==0, f"{dups} non-unique primitive paths!"

# Route B sanity: authored file must actually carry descriptions to paraphrase
if ROUTE=="B":
    n_desc = sum(1 for r in rows if str(r.get(DESC_COL,"")).strip())
    assert n_desc>0, "Route B but no DESCRIPTION present in authored CSV"
    print(f"{len(rows)} primitives | ROUTE B: paraphrasing existing masters ({n_desc} have descriptions)")
else:
    print(f"{len(rows)} primitives | ROUTE A: authoring display_name+master+{K} paraphrases")
print(f"schema: ROLE(sub-mech)={HAS_ROLE}  OBSERVABILITY_CHANNEL={HAS_CHANNEL}  SUB_TYPE={HAS_SUBTYPE}")


7480 primitives | ROUTE B: paraphrasing existing masters (7480 have descriptions)
schema: ROLE(sub-mech)=True  OBSERVABILITY_CHANNEL=False  SUB_TYPE=True


In [8]:
# resume set from side file (keyed by hash)
done = set()
if RESUME and PARA_OUT.exists():
    with PARA_OUT.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try: done.add(json.loads(line)["id"])
                except Exception: pass
print(f"already done: {len(done)}")


already done: 0


In [9]:
# ---- main loop (route decided at file level) ----
done=set()
if RESUME and PARA_OUT.exists():
    with PARA_OUT.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try: done.add(json.loads(line)["id"])
                except Exception: pass
print("already done:",len(done))

def write_rec(para_f, r, out):
    rid=pid(r)
    rec={"id":rid,"path":ppath(r),"topic":r["TOPIC"],"group":r["GROUP"],"category":r["CATEGORY"],
         "sub_type":r.get("SUB_TYPE",""),"role":r.get("ROLE",""),"channel":r.get("OBSERVABILITY_CHANNEL",""),
         "style":_style,"route":ROUTE,"paraphrases":list(out.paraphrases)}
    if ROUTE=="A":
        r[DISPLAY_COL]=out.display_name; r[DESC_COL]=out.master
        rec["display_name"]=out.display_name; rec["master"]=out.master
    else:
        rec["display_name"]=r.get(DISPLAY_COL,""); rec["master"]=r[DESC_COL]
    para_f.write(json.dumps(rec,ensure_ascii=False)+"\n"); para_f.flush()

failures=[]
para_f=PARA_OUT.open("a",encoding="utf-8")
try:
    for i,r in enumerate(rows,1):
        if pid(r) in done: continue
        try:
            out=author(r); write_rec(para_f,r,out); done.add(pid(r))
        except Exception as e:
            failures.append((ppath(r),f"{type(e).__name__}: {e}"))
            print(f"  SKIP {i}: {ppath(r)} :: {e}"); continue
        if i%25==0: print(f"{i}/{len(rows)}")
finally:
    para_f.close()
print(f"authoring complete; {len(failures)} failures")
for p,e in failures[:20]: print("  ",p,e)


already done: 0
25/7480
50/7480
75/7480
100/7480
125/7480
150/7480
175/7480
200/7480
225/7480
250/7480
275/7480
300/7480
325/7480
350/7480
375/7480
400/7480
425/7480
450/7480
475/7480
500/7480
525/7480
550/7480
575/7480
600/7480
625/7480
650/7480
675/7480
700/7480
725/7480
750/7480
775/7480
800/7480
825/7480
850/7480
875/7480
900/7480
925/7480
950/7480
975/7480
1000/7480
1025/7480
1050/7480
1075/7480
1100/7480
1125/7480
1150/7480
1175/7480
1200/7480
1225/7480
1250/7480
1275/7480
1300/7480
1325/7480
1350/7480
1375/7480
1400/7480
1425/7480
1450/7480
1475/7480
1500/7480
1525/7480
1550/7480
1575/7480
1600/7480
1625/7480
1650/7480
1675/7480
1700/7480
1725/7480
1750/7480
1775/7480
1800/7480
1825/7480
1850/7480
1875/7480
1900/7480
1925/7480
1950/7480
1975/7480
2000/7480
2025/7480
2050/7480
2075/7480
2100/7480
2125/7480
2150/7480
2175/7480
2200/7480
2225/7480
2250/7480
2275/7480
2300/7480
2325/7480
2350/7480
2375/7480
2400/7480
2425/7480
2450/7480
2475/7480
2500/7480
2525/7480
2550/7480
2575/7

### Retry failed ones

In [10]:
# ---- retry not-yet-done ----
done=set()
if PARA_OUT.exists():
    with PARA_OUT.open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): done.add(json.loads(line)["id"])
todo=[r for r in rows if pid(r) not in done]
print("remaining:",len(todo))
para_f=PARA_OUT.open("a",encoding="utf-8"); still=[]
try:
    for r in todo:
        try: out=author(r); write_rec(para_f,r,out); done.add(pid(r))
        except Exception as e: still.append((ppath(r),str(e))); print("  still failing:",ppath(r),"::",e)
finally:
    para_f.close()
print("recovered:",len(todo)-len(still),"| still failing:",len(still))


remaining: 1
recovered: 1 | still failing: 0


In [11]:
# ---- finalize ----
side={}
with PARA_OUT.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            d=json.loads(line); side[d["id"]]=d          # last write wins
missing=[ppath(r) for r in rows if pid(r) not in side]
print(f"side-file records: {len(side)} | primitives with NO record: {len(missing)}")
for m in missing[:10]: print("  ",m)

if ROUTE=="A":
    # fold authored display_name + master back into rows, then write authored CSV
    for r in rows:
        d=side.get(pid(r))
        if not d: continue
        if d.get("display_name"): r[DISPLAY_COL]=d["display_name"]
        if d.get("master"):       r[DESC_COL]=d["master"]
    with CSV_OUT.open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader(); w.writerows(rows)
    fd=sum(1 for r in rows if str(r.get(DESC_COL,"")).strip())
    print(f"ROUTE A -> wrote {CSV_OUT} | DESCRIPTION filled: {fd}/{len(rows)}")
else:
    print(f"ROUTE B -> paraphrases only in {PARA_OUT}; no CSV written (authored CSV left untouched)")


side-file records: 7480 | primitives with NO record: 0
ROUTE B -> paraphrases only in RavenPack_Edge-primitive_headline_paraphrases.jsonl; no CSV written (authored CSV left untouched)


### Check and save data

In [12]:
# ---- quick QA: template / slug-leak check on the new masters ----
import re
def slugwords(t): return set(t.replace("-", " ").split())

# leak = 0
to_show = []
for r in rows:
    if not r["DESCRIPTION"]:
        continue
    tw = slugwords(r["CATEGORY"])
    dw = set(r["DESCRIPTION"].lower().replace(".", " ").split())
    if tw and tw.issubset(dw):
        # leak += 1
        to_show.append((r["TOPIC"], r["GROUP"], r["CATEGORY"]))
print(f"masters containing full CATEGORY slug verbatim: {len(to_show)}/{len(rows)} (want ~0)")

def show(topic, group, cat):
    for r in rows:
        if (r["TOPIC"], r["GROUP"], r["CATEGORY"]) == (topic, group, cat):
            print(f"  {r['DISPLAY_NAME']:44s} | {r['DESCRIPTION']}")
print("\nSample bipolar pair:")
for t_s in to_show:
    show(*t_s)

masters containing full CATEGORY slug verbatim: 834/7480 (want ~0)

Sample bipolar pair:
  Accelerated Approval Application             | An Entity applies for an accelerated approval application for its drugs with a regulator
  Accelerated Approval Application Condition   | The accelerated approval application is filed for a drug to treat the Medical Condition
  Accelerated Approval Designation Condition   | The accelerated approval designation is granted for a drug to treat the Medical Condition
  Accelerated Approval Designation Declined Condition | The accelerated approval designation is declined for a drug to treat the Medical Condition
  Accelerated Approval Designation Declined Product | The name of the entity's Product being declined for an accelerated approval designation by a regulator
  Access To Essentials                         | The Entity is involved in giving access to essentials to a place or community
  Acquisition Merger Termination Fee           | The Company agree

---

# Taxonomy analysis

In [13]:
assert False

AssertionError: 

In [ ]:
from ravenbert.embedding import EmbeddingModel
import pandas as pd
# import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from dotenv import load_dotenv; load_dotenv()

RAVENBERT_EMBEDDING_PATH = os.environ["RAVENBERT_EMBEDDING_PATH"]

In [ ]:
embeder = EmbeddingModel.from_path(path=RAVENBERT_EMBEDDING_PATH)

# df = pl.read_csv(CSV_OUT)
df = pd.read_csv(CSV_OUT)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
E_desc = embeder.encode(df["DESCRIPTION"].tolist())
E_desc.shape

(544, 384)

In [ ]:
# scores = E_desc @ E_desc.T
# scores.shape

In [ ]:
# fig, ax = plt.subplots(figsize=(12, 10))

# sns.heatmap(
#     scores,
#     ax=ax,
#     cmap="RdYlGn",
#     # vmin=-1,
#     # vmax=1,
#     vmin=0,
#     vmax=1,
#     square=True,
#     xticklabels=False,
#     yticklabels=False,
#     cbar_kws={"shrink": 0.5},
# )

# ax.set_title("Embedding cosine similarity (1321 × 1321)")
# plt.tight_layout()
# plt.show()

In [ ]:
N_eff = 100
alpha = 0.1

1 - ( (1 - alpha)**(1/N_eff) )

0.0010530503095456112

In [ ]:
# ----------------------------------------------------------------------
# core utilities
# ----------------------------------------------------------------------

def l2norm_rows(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    return X / np.maximum(np.linalg.norm(X, axis=-1, keepdims=True), eps)


def cosine_matrix(E):
    En = l2norm_rows(E)
    C = En @ En.T
    return 0.5 * (C + C.T)


def spectrum(C):
    """Descending eigendecomposition, symmetrized for safety."""
    w, V = np.linalg.eigh(0.5 * (C + C.T))
    return w[::-1].copy(), V[:, ::-1].copy()


def participation_ratio(w, eps=1e-12):
    w = np.clip(w, 0.0, None)
    s1, s2 = w.sum(), (w**2).sum()
    return float(s1 * s1 / max(s2, eps))


def effective_rank(w, eps=1e-15):
    """exp of eigenvalue entropy; a second, entropy-flavored N_eff."""
    w = np.clip(w, 0.0, None)
    tot = w.sum()
    p = w / max(tot, eps)
    p = p[p > eps]
    return float(np.exp(-(p * np.log(p)).sum()))


def cum_explained(w):
    w = np.clip(w, 0.0, None)
    return np.cumsum(w) / w.sum()


def nearest_psd_corr(C, eps=1e-10):
    """
    Higham-style eigenvalue clipping back to the correlation cone.
    Needed for the max-8 matrix: elementwise max across pairs destroys PSD.
    Returns (C_fixed, was_modified).
    """
    C = 0.5 * (C + C.T)
    w, V = np.linalg.eigh(C)
    if w.min() >= 0:
        return C, False
    w = np.clip(w, eps, None)
    C2 = (V * w) @ V.T
    d = np.sqrt(np.diag(C2))
    return C2 / np.outer(d, d), True


def max_sim_matrix(P, block=64):
    """
    C[i,j] = max over all (paraphrase_i, paraphrase_j) pairs of cosine.
    P: (n_aspects, n_para, d). Blocked to keep peak memory ~= block*n*m*m*4 bytes
    (1321 x 8 x 8 pairs is small; block tuning is just safety).
    """
    Pn = l2norm_rows(P.reshape(-1, P.shape[-1])).reshape(P.shape).astype(np.float32)
    n, m, d = Pn.shape
    C = np.empty((n, n), dtype=np.float32)
    for i0 in range(0, n, block):
        i1 = min(i0 + block, n)
        sims = np.einsum("bkd,jld->bjkl", Pn[i0:i1], Pn)
        C[i0:i1] = sims.max(axis=(2, 3))
    return 0.5 * (C + C.T)

In [ ]:
C_raw = cosine_matrix(E_desc)
w_raw, V_raw = spectrum(C_raw)
ce_raw = cum_explained(w_raw)

print("== RAW descriptions ==")
print(f"N_eff (PR)      : {participation_ratio(w_raw):.1f} / {len(w_raw)}")
print(f"effective rank  : {effective_rank(w_raw):.1f}")
print(f"lambda1 share   : {w_raw[0]/np.clip(w_raw,0,None).sum():.3f}")
print(f"neg eigenvalues : {(w_raw < -1e-8).sum()}")
for k in [1,5,10,20,40,80,160]:
    print(f"  cum. explained @ k={k:>3} : {ce_raw[k-1]:.3f}")

== RAW descriptions ==
N_eff (PR)      : 4.5 / 544
effective rank  : 22.4
lambda1 share   : 0.464
neg eigenvalues : 0
  cum. explained @ k=  1 : 0.464
  cum. explained @ k=  5 : 0.579
  cum. explained @ k= 10 : 0.662
  cum. explained @ k= 20 : 0.754
  cum. explained @ k= 40 : 0.843
  cum. explained @ k= 80 : 0.918
  cum. explained @ k=160 : 0.973
